# Notebook 07 — Colab E2E CV Launch (VS Code Extension)
## SentinelFatal2 — End-to-End 5-Fold Cross-Validation

**הרץ נוטבוק זה דרך תוסף Google Colab ב-VS Code עם Kernel מסוג T4 GPU.**

### איך לחבר:
1. פתח נוטבוק זה ב-VS Code
2. לחץ **Select Kernel** (פינה שמאלית עליונה) → **Colab** → **New Colab Server** → בחר T4 GPU
3. הרץ תאים 1–6 לפי הסדר

| תא | פעולה | זמן |
|----|--------|-----|
| 1 | **GPU CHECK** — אשר T4 לפני המשך | שניות |
| 2 | Clone repo לשרת Colab | ~1 דקה |
| 3 | הורדת נתונים מ-GitHub אם חסרים | ~1–3 דקות |
| 4 | התקנת תלויות | ~1 דקה |
| 5 | **Dry-run** — בדוק שהכל עובד על GPU | ~5 דקות |
| 6 | **ריצה מלאה** — חוסם עד סיום (3–5 שעות) | לילה שלם |
| 7 | בדיקת תוצאות בבוקר | שניות |

> ⚠️ **השאר VS Code פתוח כל הלילה.**
> דרך התוסף, הריצה חיה כל עוד VS Code מחובר לשרת.
> אם VS Code נסגר — הריצה נעצרת. אין בעיה: הסקריפט תומך ב-resume אוטומטי — פשוט הרץ שוב את Cell 6.

In [ ]:
# ── Cell 2: Clone / Update Repository onto Colab Server ─────────────────────
# Although this notebook runs inside VS Code, the code executes on Google's
# remote server (/content/). We clone the repo there so all training scripts
# and config files are available on the remote machine.

import os, sys
from pathlib import Path

REPO_URL = "https://github.com/ArielShamay/SentinelFatal2.git"
REPO_DIR = Path("/content/SentinelFatal2")

if REPO_DIR.exists():
    print("Repository already on server — pulling latest changes...")
    os.chdir(REPO_DIR)
    print(os.popen("git pull").read())
else:
    print("Cloning repository to /content/SentinelFatal2 ...")
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed. Check internet connection.")
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

# Add repo root to Python path so 'from src.xxx import ...' works
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("✅ Repository ready on Colab server.")

In [ ]:
# ── Cell 3: Data Setup — Load Processed .npy Files from Google Drive ─────────
#
# The processed .npy files (~552 CTU-UHB + 135 FHRMA) are NOT on GitHub.
# They live locally in data/processed/ on your machine.
#
# BEFORE running this cell:
#   1. Locally, run from project root (PowerShell):
#        Compress-Archive -Path "data\processed" -DestinationPath "data_processed.zip"
#   2. Upload data_processed.zip to your Google Drive.
#   3. Update ZIP_ON_DRIVE below to the correct Drive path.
#
# For full instructions see docs/colab_e2e_cv_launch_guide.md, Section 6.

import os, sys, zipfile
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")

# ── UPDATE THIS PATH ─────────────────────────────────────────────────────────
ZIP_ON_DRIVE = "/content/gdrive/MyDrive/data_processed.zip"   # <-- change if needed
# ─────────────────────────────────────────────────────────────────────────────

ctu_dir   = REPO_DIR / "data" / "processed" / "ctu_uhb"
fhrma_dir = REPO_DIR / "data" / "processed" / "fhrma"
ctu_npy   = list(ctu_dir.glob("*.npy"))   if ctu_dir.exists()   else []
fhrma_npy = list(fhrma_dir.glob("*.npy")) if fhrma_dir.exists() else []

if len(ctu_npy) >= 552 and len(fhrma_npy) >= 135:
    print(f"✅ Processed data already present — skipping Drive mount.")
    print(f"   ctu_uhb={len(ctu_npy)}, fhrma={len(fhrma_npy)}")
else:
    print("Mounting Google Drive (auth popup may appear)...")
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=False)

    if not Path(ZIP_ON_DRIVE).exists():
        raise FileNotFoundError(
            f"\nZIP not found at: {ZIP_ON_DRIVE}\n"
            "Steps:\n"
            "  1. Locally: Compress-Archive -Path 'data\\processed' -DestinationPath 'data_processed.zip'\n"
            "  2. Upload data_processed.zip to Google Drive\n"
            "  3. Update ZIP_ON_DRIVE in this cell to match the Drive path"
        )

    print(f"Extracting {ZIP_ON_DRIVE} → {REPO_DIR / 'data'} ...")
    with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as zf:
        zf.extractall(REPO_DIR / "data")
    print("Extraction complete.")

# ── Final verification ───────────────────────────────────────────────────────
ctu_npy   = list((REPO_DIR / "data" / "processed" / "ctu_uhb").glob("*.npy"))
fhrma_npy = list((REPO_DIR / "data" / "processed" / "fhrma").glob("*.npy"))

print(f"\nVerification:")
print(f"  ctu_uhb .npy : {len(ctu_npy)}  (expected 552)")
print(f"  fhrma   .npy : {len(fhrma_npy)}  (expected 135)")

for split in ['train', 'val', 'test']:
    p = REPO_DIR / "data" / "splits" / f"{split}.csv"
    status = '✅' if p.exists() else '❌ MISSING — re-run Cell 2 (git pull)'
    print(f"  {split}.csv      : {status}")

pt_ckpt = REPO_DIR / "checkpoints" / "pretrain" / "best_pretrain.pt"
ft_ckpt = REPO_DIR / "checkpoints" / "finetune" / "best_finetune.pt"
print(f"  best_pretrain.pt : {'✅' if pt_ckpt.exists() else '⚠  not found (OK for E2E CV)'}")
print(f"  best_finetune.pt : {'✅' if ft_ckpt.exists() else '⚠  not found (OK for E2E CV)'}")

if len(ctu_npy) < 552:
    raise RuntimeError(f"Expected 552 ctu_uhb .npy files, got {len(ctu_npy)}. See Section 6 of launch guide.")
if len(fhrma_npy) < 135:
    raise RuntimeError(f"Expected 135 fhrma .npy files, got {len(fhrma_npy)}. See Section 6 of launch guide.")

print("\n✅ Data ready — safe to run Cell 4.")


In [ ]:
# ── Cell 4: Install Dependencies ─────────────────────────────────────────────
# Colab already has numpy, pandas, matplotlib, scipy, scikit-learn, torch.
# We only need to confirm PyYAML and tqdm are available.

import importlib, subprocess, sys

REQUIRED = {
    "yaml":         "PyYAML>=6.0",
    "tqdm":         "tqdm>=4.65",
    "sklearn":      "scikit-learn>=1.3",
}

to_install = []
for module, pkg in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        to_install.append(pkg)

if to_install:
    print(f"Installing: {to_install}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
else:
    print("✅ All dependencies already installed.")

# Quick sanity imports
import torch, numpy as np, pandas as pd, yaml, sklearn
print(f"\nPackage versions:")
print(f"  torch      : {torch.__version__}  (CUDA: {torch.version.cuda})")
print(f"  numpy      : {np.__version__}")
print(f"  pandas     : {pd.__version__}")
print(f"  sklearn    : {sklearn.__version__}")
print(f"  PyYAML     : {yaml.__version__}")
print(f"\n  Device     : {DEVICE}")

In [ ]:
# ── Cell 5: DRY-RUN — Verify full pipeline works on GPU (~5 min) ─────────────
# Runs 1 fold with max_batches=2. Output streams live line-by-line.
# If this cell fails, DO NOT proceed to Cell 6.

import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
os.chdir(REPO_DIR)

cmd = [
    sys.executable, "scripts/run_e2e_cv.py",
    "--device", DEVICE,
    "--dry-run",
    "--folds", "1",
    "--force-folds",
    "--seed", "42",
]

print("Starting dry-run (1 fold, max_batches=2)...", flush=True)
print("Estimated time: 3–6 minutes on T4 GPU", flush=True)
print("-" * 60, flush=True)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
proc.wait()

print("-" * 60, flush=True)
if proc.returncode == 0:
    print("✅ Dry-run PASSED — safe to run Cell 6.", flush=True)
else:
    print(f"❌ Dry-run FAILED (exit code {proc.returncode}).", flush=True)
    print("   Fix errors above before running Cell 6.", flush=True)

In [ ]:
# ── Cell 6: FULL OVERNIGHT RUN ───────────────────────────────────────────────
# This cell BLOCKS until all 5 folds complete (~3–5 hours on T4 GPU).
# Output streams live to the cell output area — you can watch fold progress.
#
# ⚠️  VS Code Colab Extension — IMPORTANT:
#   - The kernel runs on Google's server, connected through VS Code.
#   - Keep VS Code open and connected to the Colab server all night.
#   - If VS Code disconnects, the cell stops. Just re-run Cell 6 —
#     the script will automatically skip completed folds and resume.

import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
LOG_FILE = REPO_DIR / "logs" / "e2e_cv_master.log"
(REPO_DIR / "logs").mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

cmd = [
    sys.executable, "scripts/run_e2e_cv.py",
    "--device", DEVICE,
    "--force-folds",
    "--folds", "5",
    "--stride", "60",
    "--seed", "42",
]

print("=" * 60)
print("  SentinelFatal2 — Full 5-Fold E2E CV")
print(f"  Device : {DEVICE}")
print(f"  Log    : {LOG_FILE}")
print("  Keep VS Code open. This cell runs for ~3–5 hours.")
print("=" * 60)
print()

# Stream output line-by-line so progress is visible in VS Code cell output
log_lines = []
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    line_stripped = line.rstrip()
    print(line_stripped, flush=True)
    log_lines.append(line)
proc.wait()

# Save full log to disk
with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.writelines(log_lines)

print()
print("=" * 60)
if proc.returncode == 0:
    print("✅ ALL FOLDS COMPLETE — run Cell 7 to see final results.")
else:
    print(f"⚠️  Finished with exit code {proc.returncode}.")
    print("   Some folds may have failed. Run Cell 7 to check partial results.")
print("=" * 60)

In [ ]:
# ── Cell 7: MORNING CHECK — Read Results ─────────────────────────────────────
# Run this cell in the morning to check on the run.
# Can be run safely at any time — it only reads files, never modifies anything.

import os, pandas as pd
from pathlib import Path

REPO_DIR = Path("/content/SentinelFatal2")
LOG_FILE = REPO_DIR / "logs" / "e2e_cv_master.log"
PROGRESS = REPO_DIR / "logs" / "e2e_cv_progress.csv"
REPORT   = REPO_DIR / "results" / "e2e_cv_final_report.csv"

print("=" * 60)
print("  SentinelFatal2 — E2E CV Morning Check")
print("=" * 60)

# Is the process still running?
pid_check = os.popen("pgrep -f run_e2e_cv.py").read().strip()
if pid_check:
    print(f"\n  Process still running (PID: {pid_check})")
else:
    print("\n  Process has finished (or was interrupted).")

# Last 40 lines of log
print("\n-- Last 40 lines of log " + "-" * 36)
os.system(f"tail -40 {LOG_FILE}")

# Per-fold progress
print("\n-- Per-fold progress " + "-" * 39)
if PROGRESS.exists():
    df_prog = pd.read_csv(PROGRESS)
    print(df_prog.to_string(index=False))
else:
    print("  (no progress file yet — run may still be in fold 0)")

# Final report
print("\n-- Final report " + "-" * 44)
if REPORT.exists():
    df_rep = pd.read_csv(REPORT)
    print(df_rep.to_string(index=False))
    print("\n✅ Run COMPLETE.")
else:
    print("  (no final report yet — run has not completed all 5 folds)")

print("\n-- ROC curve plot " + "-" * 42)
img_path = REPO_DIR / "docs" / "images" / "e2e_cv.png"
if img_path.exists():
    from IPython.display import Image, display
    display(Image(str(img_path)))
else:
    print("  (plot not yet generated)")
print("=" * 60)